# Lekce 13 - Paměť agenta s Cognee znalostními grafy


## Nastavení

Tento zápisník ukazuje, jak vytvořit inteligentního **programovacího asistenta** s trvalou pamětí pomocí znalostních grafů [**Cognee**](https://www.cognee.ai/) a **Microsoft Agent Frameworku** (MAF).

Cognee převádí nestrukturovaný text do strukturovaného, dotazovatelného znalostního grafu podporovaného vektorovými vloženími — čímž poskytuje vašemu agentovi bohatou, vztahy uvědomující dlouhodobou paměť.

### Co se naučíte
1. **Vytvářet znalostní grafy**: Převádět profily vývojářů a osvědčené postupy do strukturovaného, dotazovatelného poznání.
2. **Integrovat Cognee s MAF**: Použít funkce `@tool` k tomu, aby agent MAF mohl dotazovat znalostní graf Cognee.
3. **Konverzace s uvědoměním sezení**: Udržovat kontext napříč několika otázkami ve stejném sezení.
4. **Dlouhodobá paměť**: Uchovávat důležité poznatky mezi sezeními a získávat je v nových rozhovorech.

### Požadavky
- Python 3.9 a novější
- Redis spuštěný lokálně (`docker run -d -p 6379:6379 redis`) pro správu sezení
- API klíč LLM (např. OpenAI) — nastavit `LLM_API_KEY` v `.env`
- `CACHING=true` v `.env` (vyžadováno pro sezení Cognee)
- Projekt Azure AI Foundry s nasazeným chat modelem
- `AZURE_AI_PROJECT_ENDPOINT` a `AZURE_AI_MODEL_DEPLOYMENT_NAME` v `.env`
- Přihlášený Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typy paměti agenta

Tento sešit zkoumá stejné tři typy paměti jako hlavní sešit Lekce 13, ale používá Cognee jako backend pro dlouhodobou paměť:

| Typ paměti | Mechanismus | Doba trvání |
|---|---|---|
| **Pracovní** | `agent.create_session()` (MAF) | Jedna konverzace |
| **Krátkodobá** | Cache relace Cognee (Redis) | Jedna relace |
| **Dlouhodobá** | Cognee knowledge graph + vektory | Trvalá |

### Architektura paměti Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Připravte úložiště Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Část 1 — Vytváření znalostní báze

Zpracováváme tři typy dat pro vytvoření komplexní znalostní báze pro našeho kódovacího asistenta:

1. **Profil vývojáře** — osobní odborné znalosti a technické zázemí  
2. **Nejlepší praktiky Pythonu** — Zen Pythonu s praktickými pokyny  
3. **Historické konverzace** — minulá sezení otázek a odpovědí mezi vývojáři a AI asistenty


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizace znalostního grafu

Cognee může vykreslit interaktivní HTML vizualizaci entit a vztahů, které extrahoval.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Obohaťte paměť pomocí Memify

`memify()` analyzuje znalostní graf a vytváří inteligentní pravidla — identifikuje vzorce, osvědčené postupy a vztahy mezi koncepty.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Část 2 — MAF agent s nástroji Cognee

Nyní vytvoříme MAF agenta, který může dotazovat znalostní graf Cognee pomocí funkcí `@tool`. To umožňuje agentovi využít plnou sílu sémantického vyhledávání založeného na grafu při zachování kontextu konverzace prostřednictvím relací.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Pracovní paměť se sezeními

`AgentSession` (vytvořený pomocí `agent.create_session()`) poskytuje pracovní paměť v rámci sezení. Agent se může odkazovat na předchozí zprávy a zároveň dotazovat dlouhodobý znalostní graf Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nová relace — dlouhodobá paměť přetrvává

Zahájení nové relace vymaže pracovní paměť, ale Cognee znalostní graf je stále dostupný. Agent může získat stejné dlouhodobé znalosti v úplně novém rozhovoru.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Shrnutí

V tomto sešitu jste vytvořili kódovacího asistenta, který kombinuje **pracovní paměť MAF** (`agent.create_session()`) s **dlouhodobým znalostním grafem Cognee**.

### Co jste se naučili
1. **Konstrukce znalostního grafu**: Cognee zpracovává nestrukturovaný text a vytváří graf + vektorovou paměť.
2. **Obohacení grafu pomocí memify**: Odvozené fakta a bohatší vztahy nad existujícím grafem.
3. **Integrace MAF + Cognee**: Funkce `@tool` umožňují agentovi MAF přirozeně dotazovat se v grafu Cognee.
4. **Pracovní paměť + dlouhodobá paměť**: `AgentSession` (pomocí `agent.create_session()`) poskytuje kontext relace, zatímco Cognee poskytuje trvalé znalosti.
5. **Filtrované vyhledávání pomocí NodeSets**: Cílení na specifické podmnožiny znalostního grafu (např. pouze principy).

### Hlavní poznatky
- **Cognee** převádí surový text do strukturované, vztahy uvědomělé paměti — výkonnější než jednoduché vektorové úložiště.
- **Funkce `@tool`** čistě propojují agenty MAF a externí znalostní systémy.
- **`AgentSession`** (pomocí `agent.create_session()`) udržuje kontext na úrovni konverzace oddělený od dlouhodobých znalostí.
- Stejný znalostní graf slouží více relacím a agentům.

### Praktické využití
- **Vývojářští kopiloti**: Revize kódu, analýza incidentů, asistenti architektury
- **Kopiloti pro zákazníky**: Podpora přes dokumentaci produktů, FAQ a poznámky CRM
- **Interní expertní kopiloti**: Právní, bezpečnostní či politické asistenty pracující s pravidly
- **Unifikované datové vrstvy**: Kombinace strukturovaných a nestrukturovaných dat do jednoho dotazovatelného grafu

### Další kroky
- Experimentovat s časovou povědomostí v Cognee
- Definovat OWL ontologii pro doménově specifickou kvalitu grafu
- Přidat zpětnou vazbu uživatelů pro zlepšení vyhledávání v čase
- Škálovat na multi-agentní systémy sdílející stejnou paměťovou vrstvu Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Ačkoli usilujeme o přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za závazný zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nezodpovídáme za jakékoliv nedorozumění nebo nesprávné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
